**ALISTAMIENTO DEL ENTORNO E IMPORTACIÓN DE LIBRERÍAS**

In [ ]:
# ==============================================================================
# IMPORTACIÓN DE LIBRERÍAS CORE
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from transformers import pipeline
from sklearn.linear_model import LinearRegression
from matplotlib.backends.backend_pdf import PdfPages
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')

# ==============================================================================
# PASO 6: NLP - ANÁLISIS DE SENTIMIENTO (MINERÍA DE TEXTO)
# ==============================================================================
print("🧠 Iniciando Paso 6: Procesamiento de Lenguaje Natural (NLP)...")

# Carga de la base de Scopus
df_scopus = pd.read_csv('scopus.csv', sep=',', encoding='utf-8')
col_abstract = [col for col in df_scopus.columns if 'Abstract' in col][0]
df_scopus_clean = df_scopus.dropna(subset=['Year', col_abstract])

# Motor de NLP
nlp_sentiment = pipeline("sentiment-analysis", truncation=True, max_length=512)

def extraer_sentimiento(texto):
    try:
        return nlp_sentiment(str(texto))[0]['label']
    except:
        return "NEUTRAL"

df_scopus_clean['Sentimiento'] = df_scopus_clean[col_abstract].apply(extraer_sentimiento)
kpi_sentimiento = df_scopus_clean['Sentimiento'].value_counts(normalize=True) * 100

porcentaje_pos = np.round(kpi_sentimiento.get('POSITIVE', 0), 1)
porcentaje_neg = np.round(kpi_sentimiento.get('NEGATIVE', 0), 1)

# ==============================================================================
# PASO 7: ML Y ESTADÍSTICA (SERIES DE TIEMPO Y FUNCIÓN DE DENSIDAD PDF)
# ==============================================================================
print("⚙️ Iniciando Paso 7: Machine Learning y Estadística (PDF)...")

tendencia = df_scopus_clean.groupby('Year').size().reset_index(name='Publicaciones')
tendencia = tendencia[tendencia['Year'] >= 2010]

# 1. Machine Learning (Regresión Lineal)
X = tendencia[['Year']]
y = tendencia['Publicaciones']
modelo_tendencia = LinearRegression()
modelo_tendencia.fit(X, y)

año_actual = 2026
años_futuros = pd.DataFrame({'Year': [año_actual, año_actual + 1, año_actual + 2]})
años_futuros['Proyeccion'] = np.round(modelo_tendencia.predict(años_futuros)).astype(int)

# 2. Estadística: Función de Densidad de Probabilidad (PDF)
media = np.mean(y)
desviacion_estandar = np.std(y)
x_pdf = np.linspace(min(y), max(y) + 500, 100)
y_pdf = norm.pdf(x_pdf, media, desviacion_estandar)

# ==============================================================================
# PASOS 8, 9 Y 10: GENERACIÓN DEL DASHBOARD ESTRATÉGICO
# ==============================================================================
print("📊 Construyendo Dashboard y Reporte Estratégico...")

archivo_salida = '/content/Dashboard_Inteligencia_Negocios.pdf'

with PdfPages(archivo_salida) as pdf:
    # --- PÁGINA 1: VISUALIZACIÓN DE DATOS ---
    fig1 = plt.figure(figsize=(12, 10))
    fig1.suptitle('Dashboard Analítico: Modelos Predictivos en Salud Pública', fontsize=16, fontweight='bold')

    # Gráfico A: Proyección ML
    ax1 = plt.subplot(2, 2, 1)
    ax1.plot(años_futuros['Year'].astype(str), años_futuros['Proyeccion'], marker='o', color='#1f77b4', linewidth=2)
    ax1.set_title('Machine Learning: Proyección de Publicaciones')
    ax1.set_ylabel('Volumen Proyectado')
    for i, txt in enumerate(años_futuros['Proyeccion']):
        ax1.annotate(txt, (str(años_futuros['Year'].iloc[i]), años_futuros['Proyeccion'].iloc[i]), textcoords="offset points", xytext=(0,10), ha='center')

    # Gráfico B: Estadística PDF (Función de Densidad)
    ax2 = plt.subplot(2, 2, 2)
    ax2.plot(x_pdf, y_pdf, color='#ff7f0e', linewidth=2)
    ax2.fill_between(x_pdf, y_pdf, alpha=0.3, color='#ff7f0e')
    ax2.set_title('Estadística (PDF): Densidad de Probabilidad')
    ax2.set_ylabel('Densidad')

    # Gráfico C: Sentimiento NLP
    ax3 = plt.subplot(2, 1, 2)
    categorias = ['Aceptación Técnica', 'Riesgo Ético / Sesgos']
    ax3.barh(categorias, [porcentaje_pos, porcentaje_neg], color=['#2ca02c', '#d62728'])
    ax3.set_title('NLP: Sentimiento de la Literatura Científica')
    for i, v in enumerate([porcentaje_pos, porcentaje_neg]):
        ax3.text(v + 1, i, f"{v}%", color='black', va='center', fontweight='bold')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    pdf.savefig(fig1)
    plt.close()

    # --- PÁGINA 2: ESTRATEGIA ORGANIZACIONAL ---
    fig2 = plt.figure(figsize=(12, 8))
    plt.axis('off')

    texto_llm = f"""
    REPORTE LLM Y PLAN ESTRATÉGICO DE NEGOCIOS (PASOS 8, 9 Y 10)

    1. Interpretación Analítica (Paso 8):
    La Función de Densidad de Probabilidad (PDF) y el modelo de Machine Learning demuestran un
    crecimiento sostenido en la implementación de analítica para poblaciones vulnerables, proyectando
    {años_futuros['Proyeccion'].iloc[-1]} publicaciones para {años_futuros['Year'].iloc[-1]}.
    No obstante, el NLP revela un {porcentaje_neg}% de literatura enfocada en riesgos éticos.

    2. Estrategias de Negocio (Paso 9):
    Para mitigar estos riesgos en arquitecturas organizacionales y bases de datos poblacionales,
    la estrategia central debe enfocarse en garantizar la explicabilidad y la confianza en la
    adopción de algoritmos.
    - Implementar marcos de trabajo de "IA Explicable" (XAI) que permitan a los auditores
      trazar cómo el modelo predictivo asigna o restringe recursos.
    - Establecer protocolos de equidad algorítmica (Fairness) en la fase de entrenamiento.

    3. Diseño de KPIs (Paso 10):
    - Perspectiva de Aprendizaje: Índice de cobertura de auditoría algorítmica (Meta: 100%).
    - Perspectiva de Procesos: Nivel de interpretabilidad del modelo predictivo (Meta: >85%).
    - Perspectiva del Cliente/Usuario: Tasa de reclamaciones por decisiones automatizadas
      sesgadas (Meta: <2%).
    """
    plt.text(0.05, 0.9, texto_llm, fontsize=11, family='monospace', va='top', wrap=True)
    pdf.savefig(fig2)
    plt.close()

print("✅ Ejecución completada. El archivo 'Dashboard_Inteligencia_Negocios.pdf' está listo para descarga.")

🧠 Iniciando Paso 6: Procesamiento de Lenguaje Natural (NLP)...


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]